# EDA

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
df = pd.read_csv("/kaggle/input/titanic/train.csv")
df.head()

In [ ]:
df = df.drop(['PassengerId','Name','Ticket','Cabin'] , axis='columns')

In [ ]:
df.info()

In [ ]:
df.dropna(subset=['Embarked'], inplace=True)

In [ ]:
plt.figure(figsize=(8,6))
plt.hist(df['Age'], bins = 50)
plt.title("Age Distribution")
plt.xlabel('Age')
plt.ylabel('Frequency')

In [ ]:
sns.histplot(data=df, x="Fare")

# Data preprocessing

In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

In [ ]:
#Encoding categorical values
encoder = LabelEncoder()
df['Sex'] = encoder.fit_transform(df['Sex'])
df['Embarked'] = encoder.fit_transform(df['Embarked'])

In [ ]:
scaler = MinMaxScaler()
df[['Age', 'Fare']] = scaler.fit_transform(df[['Age', 'Fare']])

# Replace NaN values in Age

In [ ]:
columns = df.columns
columns

In [ ]:
from sklearn.impute import KNNImputer

In [ ]:
imputer = KNNImputer(n_neighbors=5)
df = imputer.fit_transform(df)

In [ ]:
df = pd.DataFrame(df, columns= columns)

In [ ]:
plt.figure(figsize=(8,6))
plt.hist(df['Age'], bins = 50)
plt.title("Age Distribution")
plt.xlabel('Age')
plt.ylabel('Frequency')

# ML Models

In [ ]:
X = df.iloc[ : , 1:]
y = df.iloc[ : , 0]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2)

# LogisticRegression

In [ ]:
logreg = LogisticRegression()
logreg.fit(X_train, y_train)
acc_log = round(logreg.score(X_train, y_train) * 100, 3)
log = round(logreg.score(X_test, y_test) * 100, 3)
print("Train: ", acc_log)
print("Test: ", log)

# RandomForestClassifier

In [ ]:
RandomForestClassifier = RandomForestClassifier(
    max_features=.7,
    min_samples_leaf=10,
    min_samples_split=8,
    n_estimators=10,
    max_depth=15,
    verbose=1)
RandomForestClassifier.fit(X_train, y_train)
acc_random_forest = round(RandomForestClassifier.score(X_train, y_train) * 100, 3) 
random_forest = round(RandomForestClassifier.score(X_test, y_test) * 100, 3) 
print("Train: ", acc_random_forest)
print("Test: ", random_forest)

# GradientBoostingClassifier

In [ ]:
GradientBoostingClassifier = GradientBoostingClassifier(
    learning_rate=0.01,
    max_depth=10,
    max_features=0.8,
    min_samples_leaf=2,
    min_samples_split=2,
    n_estimators=100,
    subsample=0.4,
    random_state=42
)
GradientBoostingClassifier.fit(X_train, y_train)
acc_GBC = round(GradientBoostingClassifier.score(X_train, y_train) * 100, 3)
GBC = round(GradientBoostingClassifier.score(X_test, y_test) * 100, 3)
print("Train: ", acc_GBC)
print("Test: ", GBC)

# ExtraTreesClassifier

In [ ]:
ExtraTreesClassifier = ExtraTreesClassifier(
    bootstrap=True,
    criterion='gini',
    max_features=0.65,
    min_samples_leaf=1,
    min_samples_split=6,
    n_estimators=100,
    random_state=42
)
ExtraTreesClassifier.fit(X_train, y_train)
acc_extra_tree = round(ExtraTreesClassifier.score(X_train, y_train) * 100, 3)
extra_tree = round(ExtraTreesClassifier.score(X_test, y_test) * 100, 3)
print("Train: ", acc_extra_tree)
print("Test: ", extra_tree)

## Confusion matrix to show how many times the model went wrong.

In [ ]:
y_predicted = ExtraTreesClassifier.predict(X_test)
cm = confusion_matrix(y_test, y_predicted)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True,cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Truth')

In [ ]:
models = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest',
              'ExtraTreesClassifier','GradientBoostingClassifier'],
    
    'Training_score': [acc_log, acc_random_forest,
                       acc_extra_tree,acc_GBC],
    
    'Testing_score' : [log, random_forest,
                       extra_tree,GBC]})

models.sort_values(by='Testing_score', ascending=False)

# Submission

In [ ]:
test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
test = test_data.drop(['PassengerId','Name','Ticket','Cabin'] , axis='columns')
test['Sex'] = encoder.fit_transform(test['Sex'])
test['Embarked'] = encoder.fit_transform(test['Embarked'])
imputer = KNNImputer(n_neighbors=2)
test = imputer.fit_transform(test)
test = pd.DataFrame(test, columns=columns[1:])
predictions = RandomForestClassifier.predict(test)
predictions = predictions.astype(int)

In [ ]:
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('/kaggle/working/submission.csv', index=False)

# Save the model

In [ ]:
import pickle

In [ ]:
with open('Titanic_model','wb') as file:
    pickle.dump(GradientBoostingClassifier,file)